# learn-to-remember: behavior analysis (M6)

Reads the eval result files and training logs listed in `configs/report.yaml` and
draws the figures and RQ tables. Every number comes from a result file; a run
whose file is missing is shown as **not run**, never filled in. No GPU or model
is needed: copy each Kaggle run's result folder into `results/` first.

The same code backs `scripts/make_figures.py` and `scripts/make_report.py`, so
this notebook and the committed figures cannot disagree.

In [ ]:
import os, sys

# Repo root: this notebook lives in notebooks/. LTR_* variables override the
# defaults (the test suite runs this notebook on synthetic results that way).
REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(REPO, "src"))
ROOT = os.environ.get("LTR_RESULTS_ROOT", REPO)
SPEC_PATH = os.environ.get("LTR_SPEC", os.path.join(REPO, "configs", "report.yaml"))
FIG_DIR = os.environ.get("LTR_FIG_DIR", os.path.join(REPO, "figures"))

from IPython.display import Image, Markdown, display
from learn_to_remember import analysis as an
from learn_to_remember.figures import make_all
from learn_to_remember.model_utils import load_config
from learn_to_remember.report import build_report

spec = load_config(SPEC_PATH)
print("results root:", ROOT)
print("model slug:  ", spec.get("slug"))

## 1. Which result files exist

One row per eval file the report reads. Anything marked missing is "not run".

In [ ]:
avail = an.availability(spec, ROOT)
for run, company, cond, ok in avail:
    print(f"{'ok         ' if ok else 'not run    '} {run:13s} {company:10s} {cond}")
print(f"\n{sum(ok for *_, ok in avail)} of {len(avail)} eval files present")

## 2. Figures

Written to `figures/` (plus `figures/INDEX.md`). Error bars are 95% bootstrap CIs
over questions. Colors follow the entity in every figure: base model blue,
trained on v3 notes orange, trained on v0 notes aqua.

In [ ]:
res = make_all(spec, ROOT, FIG_DIR)
for name, path in res["written"].items():
    display(Markdown(f"**{name}**"))
    display(Image(filename=path))
if res["not_run"]:
    print("not run (no result files):", ", ".join(res["not_run"]))

## 3. RQ tables

Paired differences on the same questions: bootstrap 95% CI on the difference and
an exact McNemar p for accuracy. A difference counts only when its CI excludes 0.

In [ ]:
display(Markdown(build_report(spec, ROOT)))

## 4. Paired examples: base vs trained (λ=0.2), company A, notes v3

Questions the trained model got right and the base got wrong ("wins") and the
reverse ("losses"). The pick is fixed (first by qtype order, then id), not
chosen by hand. Full trajectories need the `rollouts/` folders (gitignored);
otherwise only the saved samples are available.

In [ ]:
ex = an.paired_examples(spec, ROOT, k=2)
if ex["missing"]:
    print("not run:", ", ".join(ex["missing"]))
else:
    print(f"wins: {ex['n_wins']}   losses: {ex['n_losses']}")
    for kind in ("wins", "losses"):
        for e in ex[kind]:
            print(f"\n==================== {kind[:-1]}: {e['id']} ({e['qtype']})")
            print("---------- base"); print(an.format_trajectory(e["base"]))
            print("---------- trained"); print(an.format_trajectory(e["notes_l0.2"]))

## 5. Reading the results

Not written yet: no GPU run has finished (M3-M5 are "not run"). Once they have,
this section and the README state, for each RQ, what the CIs support and what
they do not, including negative and null results, in plain words.